In [0]:
%run "../SetUp/setup" 

### Access Azure Data Lake using Service Principal
**Steps to follow:**
1. Register Azure AD Application/ Service Principal
2. Generate a secret/ password for the application
3. Set spark config with App/ Client Id, Directory/ Tenant Id & Secret
4. Assign role "Storage Blob Data Contributor" to the Data Lake

path,name,size,modificationTime
abfss://raw@forrmulaa1dl.dfs.core.windows.net/circuits.csv,circuits.csv,10044,1767877118000
abfss://raw@forrmulaa1dl.dfs.core.windows.net/constructors.json,constructors.json,30415,1767877118000
abfss://raw@forrmulaa1dl.dfs.core.windows.net/drivers.json,drivers.json,180812,1767877118000
abfss://raw@forrmulaa1dl.dfs.core.windows.net/lap_times/,lap_times/,0,1767877145000
abfss://raw@forrmulaa1dl.dfs.core.windows.net/pit_stops.json,pit_stops.json,1369387,1767877119000
abfss://raw@forrmulaa1dl.dfs.core.windows.net/qualifying/,qualifying/,0,1767877183000
abfss://raw@forrmulaa1dl.dfs.core.windows.net/races.csv,races.csv,116847,1767877118000
abfss://raw@forrmulaa1dl.dfs.core.windows.net/results.json,results.json,7165641,1767877120000


_c0,_c1,_c2,_c3,_c4,_c5,_c6,_c7,_c8
circuitId,circuitRef,name,location,country,lat,lng,alt,url
1,albert_park,Albert Park Grand Prix Circuit,Melbourne,Australia,-37.8497,144.968,10,http://en.wikipedia.org/wiki/Melbourne_Grand_Prix_Circuit
2,sepang,Sepang International Circuit,Kuala Lumpur,Malaysia,2.76083,101.738,18,http://en.wikipedia.org/wiki/Sepang_International_Circuit
3,bahrain,Bahrain International Circuit,Sakhir,Bahrain,26.0325,50.5106,7,http://en.wikipedia.org/wiki/Bahrain_International_Circuit
4,catalunya,Circuit de Barcelona-Catalunya,Montmeló,Spain,41.57,2.26111,109,http://en.wikipedia.org/wiki/Circuit_de_Barcelona-Catalunya
5,istanbul,Istanbul Park,Istanbul,Turkey,40.9517,29.405,130,http://en.wikipedia.org/wiki/Istanbul_Park
6,monaco,Circuit de Monaco,Monte-Carlo,Monaco,43.7347,7.42056,7,http://en.wikipedia.org/wiki/Circuit_de_Monaco
7,villeneuve,Circuit Gilles Villeneuve,Montreal,Canada,45.5,-73.5228,13,http://en.wikipedia.org/wiki/Circuit_Gilles_Villeneuve
8,magny_cours,Circuit de Nevers Magny-Cours,Magny Cours,France,46.8642,3.16361,228,http://en.wikipedia.org/wiki/Circuit_de_Nevers_Magny-Cours
9,silverstone,Silverstone Circuit,Silverstone,UK,52.0786,-1.01694,153,http://en.wikipedia.org/wiki/Silverstone_Circuit


In [0]:
%run "../Includes/configs" 

**Produce Constructor Standings**

In [0]:
race_results_df = spark.read.parquet(f"{presentation_folder_path}/race_results")

In [0]:
display(race_results_df)

race_year,race_name,race_date,circuit_location,driver_name,driver_number,driver_nationality,team,grid,fastest_lap,race_time,points,position,created_date
2018,Australian Grand Prix,2018-03-25T05:10:00Z,Melbourne,Sergey Sirotkin,35,Russian,Williams,19,3,\N,0.0,null,2026-01-12T15:36:32.584276Z
2018,Australian Grand Prix,2018-03-25T05:10:00Z,Melbourne,Marcus Ericsson,9,Swedish,Sauber,17,4,\N,0.0,null,2026-01-12T15:36:32.584276Z
2018,Australian Grand Prix,2018-03-25T05:10:00Z,Melbourne,Pierre Gasly,10,French,Toro Rosso,20,13,\N,0.0,null,2026-01-12T15:36:32.584276Z
2018,Australian Grand Prix,2018-03-25T05:10:00Z,Melbourne,Kevin Magnussen,20,Danish,Haas F1 Team,5,21,\N,0.0,null,2026-01-12T15:36:32.584276Z
2018,Australian Grand Prix,2018-03-25T05:10:00Z,Melbourne,Romain Grosjean,8,French,Haas F1 Team,6,23,\N,0.0,null,2026-01-12T15:36:32.584276Z
2018,Australian Grand Prix,2018-03-25T05:10:00Z,Melbourne,Brendon Hartley,28,New Zealander,Toro Rosso,16,57,\N,0.0,15,2026-01-12T15:36:32.584276Z
2018,Australian Grand Prix,2018-03-25T05:10:00Z,Melbourne,Lance Stroll,18,Canadian,Williams,13,55,+1:18.288,0.0,14,2026-01-12T15:36:32.584276Z
2018,Australian Grand Prix,2018-03-25T05:10:00Z,Melbourne,Charles Leclerc,16,Monegasque,Sauber,18,56,+1:15.759,0.0,13,2026-01-12T15:36:32.584276Z
2018,Australian Grand Prix,2018-03-25T05:10:00Z,Melbourne,Esteban Ocon,31,French,Force India,14,57,+1:00.278,0.0,12,2026-01-12T15:36:32.584276Z
2018,Australian Grand Prix,2018-03-25T05:10:00Z,Melbourne,Sergio Pérez,11,Mexican,Force India,12,51,+46.817,0.0,11,2026-01-12T15:36:32.584276Z


In [0]:
from pyspark.sql.functions import sum, count, when, col

In [0]:
constructor_standings_df = race_results_df.groupBy("race_year", "team").agg(sum("points").alias("total_points"), count(when(col("position") == 1, True)).alias("wins"))

In [0]:
display(constructor_standings_df.filter("race_year = 2020"))

race_year,team,total_points,wins
2020,Haas F1 Team,3.0,0
2020,McLaren,202.0,0
2020,Ferrari,131.0,0
2020,Mercedes,573.0,13
2020,AlphaTauri,107.0,1
2020,Williams,0.0,0
2020,Red Bull,319.0,2
2020,Alfa Romeo,8.0,0
2020,Racing Point,210.0,1
2020,Renault,181.0,0


In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import desc, rank, asc 

In [0]:
constructor_rank_spec = Window.partitionBy("race_year").orderBy(desc("total_points"), desc("wins"))
final_df = constructor_standings_df.withColumn("rank", rank().over(constructor_rank_spec))

In [0]:
display(final_df.filter("race_year == 2020"))

race_year,team,total_points,wins,rank
2020,Mercedes,573.0,13,1
2020,Red Bull,319.0,2,2
2020,Racing Point,210.0,1,3
2020,McLaren,202.0,0,4
2020,Renault,181.0,0,5
2020,Ferrari,131.0,0,6
2020,AlphaTauri,107.0,1,7
2020,Alfa Romeo,8.0,0,8
2020,Haas F1 Team,3.0,0,9
2020,Williams,0.0,0,10


In [0]:
final_df.write.mode("overwrite").parquet(f"{presentation_folder_path}/constructor_standings")